#### LangSmith

LangSmith is a platform for observability, debugging, evaluation, and monitoring of LLM applications and agents.

Langsmith is langchain's Observability, debugging, and evaluation platform purpose-built for LLM applications. while OpenTelemetry is the generic plumbing, langSmith is the specialized UI and backend that understands Langchain and Langgraph natively -- no manual span creation needed for most cases.

It works particularly well with:
* LangGraph
* Langchain
* LLM Calls
* Tools
* Retriveres
* Agent Workflows



##### Mental Model of the LangSmith

``` text
LangGraph Agent
      ↓  (automatic — zero code change)
LangSmith SDK
      ↓
LangSmith Cloud / Self-hosted
      ↓
┌──────────────────────────────────┐
│  Traces   Runs    Datasets       │
│  Spans    Tags    Evaluations    │
│  Metadata Feedback Experiments   │
└──────────────────────────────────┘
      ↓
Find bottleneck → Find failed node → Inspect I/O → Fix
```

**Key difference from generic OTel**: LangSmith understands prompts, completions, token costs, tool calls, and retriever results semantically — not just as arbitrary key-value attributes.

2. Core Concepts

2.1 Projects

A Project is a logical namespace. Every trace goes into a project.

``` markdown
Projects
├── hdfc_credit_card_bot         ← production
├── reliance_support_agent_dev   ← development
├── zomato_recommendation_agent  ← staging
└── experiments_subbu            ← personal experiments
```

2.2 Runs

A Run is the record of a single invocation of any LangChain/LangGraph component. Every node execution, every LLM call, every tool call = one Run.

Runs have a type:
``` markdown
Run Type            What it records
chain               A Langgraph Node, an LCEL chain, any custom Logic
llm                 A Model call(input Prompt, output, token counts,cost)
tool                A tool invocation (input args, output)
retriever           A vector store / retriever call (query, returned docs)
embedding           An Embedding generation call
````

2.3 Traces

A Trace = the full tree of Runs for one end-to-end request. It has a root Run (the graph invocation) with child Runs branching off it.

``` markdown
Trace: handle_customer_complaint  [root Run — type: chain]
├── supervisor_node               [Run — type: chain]
│   └── gpt-4o call               [Run — type: llm]  ← prompt + completion + tokens
├── billing_agent                 [Run — type: chain]
│   ├── gpt-4o call               [Run — type: llm]
│   └── get_invoice               [Run — type: tool]  ← args + result
└── response_formatter            [Run — type: chain]
    └── gpt-4o call               [Run — type: llm]
```

2.4 Spans

In LangSmith's UI, each Run is rendered as a Span in the waterfall view — a horizontal bar showing start time, duration, and type. This is the same concept as OTel spans, automatically populated.

2.5 Metadata

Metadata = arbitrary key-value dict you attach to any Run. Useful for business context:

``` python
{"customer_id": "CUST-10247", "region": "Maharashtra", "channel": "WhatsApp"}
```

2.6 Tags

Tags = list of string labels on a Run. Used for filtering and grouping:

``` python
["production", "hindi_query", "escalated", "refund_flow"]
```

3. Setup — Enabling LangSmith in 3 Lines

``` bash
pip install langsmith langchain langgraph
```

``` bash
# .env
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__your_api_key_here
LANGCHAIN_PROJECT=reliance_support_agent_dev
```

``` python 
# main.py — that's it. LangGraph traces automatically.
from dotenv import load_dotenv
load_dotenv()
```

4. LangGraph Tracing — What You Get Automatically

Let's build a realistic HDFC credit card support agent and see exactly what LangSmith captures.


In [ ]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator

load_dotenv()

# ─── State ───────────────────────────────────────────────────────────────────

class CardSupportState(TypedDict):
    customer_id: str
    query: str
    query_type: str          # "billing" | "rewards" | "block_card"
    resolved: bool
    response: str
    messages: Annotated[list, operator.add]

llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

# ─── Nodes ───────────────────────────────────────────────────────────────────

def classify_query(state: CardSupportState) -> CardSupportState:
    """Supervisor: classify the customer's query."""
    response = llm.invoke([
        SystemMessage(content="""
            You are an HDFC credit card support classifier.
            Classify the query into exactly one of: billing, rewards, block_card.
            Respond with only the category word.
        """),
        HumanMessage(content=state["query"])
    ])
    query_type = response.content.strip().lower()
    return {**state, "query_type": query_type, "messages": [f"Classified: {query_type}"]}


def handle_billing(state: CardSupportState) -> CardSupportState:
    """Handle billing and payment queries."""
    response = llm.invoke([
        SystemMessage(content="""
            You are an HDFC Bank billing specialist. 
            Answer billing questions concisely. Always mention EMI options when relevant.
            Amounts in INR. Keep response under 3 sentences.
        """),
        HumanMessage(content=state["query"])
    ])
    return {**state, "response": response.content, "resolved": True}


def handle_rewards(state: CardSupportState) -> CardSupportState:
    """Handle rewards and cashback queries."""
    response = llm.invoke([
        SystemMessage(content="""
            You are an HDFC SmartBuy rewards specialist.
            Answer rewards/cashback questions. Mention reward point value: 1 point = ₹0.25.
            Keep response under 3 sentences.
        """),
        HumanMessage(content=state["query"])
    ])
    return {**state, "response": response.content, "resolved": True}


def handle_block_card(state: CardSupportState) -> CardSupportState:
    """Handle card blocking requests."""
    response = llm.invoke([
        SystemMessage(content="""
            You are an HDFC card security specialist.
            For block card requests, confirm the block and provide the hotline: 1800-202-6161.
            Express urgency. Keep response under 3 sentences.
        """),
        HumanMessage(content=state["query"])
    ])
    return {**state, "response": response.content, "resolved": True}


def route_query(state: CardSupportState) -> Literal["billing", "rewards", "block_card"]:
    return state["query_type"]

# ─── Graph ───────────────────────────────────────────────────────────────────

graph = StateGraph(CardSupportState)
graph.add_node("classify", classify_query)
graph.add_node("billing", handle_billing)
graph.add_node("rewards", handle_rewards)
graph.add_node("block_card", handle_block_card)

graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_query, {
    "billing": "billing",
    "rewards": "rewards",
    "block_card": "block_card"
})
graph.add_edge("billing", END)
graph.add_edge("rewards", END)
graph.add_edge("block_card", END)

app = graph.compile()

# ─── Invoke ──────────────────────────────────────────────────────────────────

result = app.invoke({
    "customer_id": "HDFC-CUST-98234",
    "query": "My credit card statement shows a charge of ₹8,500 from Swiggy that I didn't make.",
    "query_type": "",
    "resolved": False,
    "response": "",
    "messages": []
})

print(result["response"])

What LangSmith captures automatically (zero extra code):

``` markdown
Trace: LangGraph
├── classify                    [chain]  500ms
│   └── ChatOpenAI              [llm]    480ms
│       ├── Input:  system + human messages
│       ├── Output: "billing"
│       └── Tokens: prompt=89, completion=1, cost=$0.00027
├── billing                     [chain]  720ms
│   └── ChatOpenAI              [llm]    700ms
│       ├── Input:  system + human messages
│       ├── Output: "We have flagged this ₹8,500..."
│       └── Tokens: prompt=112, completion=54, cost=$0.0019
```

Every prompt, every completion, every token count, every cost — captured.

5. Adding Metadata and Tags

Metadata and tags make your traces searchable and filterable in the LangSmith UI.

In [ ]:
from langchain_core.runnables.config import RunnableConfig

result = app.invoke(
    {
        "customer_id": "HDFC-CUST-98234",
        "query": "My credit card statement shows ₹8,500 from Swiggy I didn't make.",
        "query_type": "",
        "resolved": False,
        "response": "",
        "messages": []
    },
    config=RunnableConfig(
        # Tags — for grouping/filtering
        tags=["production", "fraud_query", "maharashtra", "v2.1"],
        
        # Metadata — arbitrary business context
        metadata={
            "customer_id": "HDFC-CUST-98234",
            "customer_tier": "Regalia",
            "channel": "mobile_app",
            "city": "Pune",
            "session_id": "sess_8823kl",
            "ab_test_variant": "B"
        }
    )
)

Now in Langsmith we can filter:
* All runs tagged fraud_query
* All runs where customer_tier = Regalia
* All runs from city = Pune

5.2 At the Node Level — get_current_run_tree

For node-specific metadata (more granular than the root trace):

In [1]:
from langsmith.run_helpers import get_current_run_tree

def handle_billing(state: CardSupportState) -> CardSupportState:
    # Attach metadata to THIS node's run specifically
    run = get_current_run_tree()
    if run:
        run.extra = run.extra or {}
        run.extra["metadata"] = {
            "dispute_amount_inr": 8500,
            "merchant": "Swiggy",
            "escalation_required": True
        }
    
    response = llm.invoke([...])
    return {**state, "response": response.content, "resolved": True}

KeyboardInterrupt: 

6. Custom Instrumentation — @traceable

For code that isn't LangChain/LangGraph (your own Python functions), use @traceable to create Runs manually.

In [ ]:
from langsmith import traceable

@traceable(
    run_type="tool",
    name="fetch_hdfc_statement",
    tags=["database", "billing"],
    metadata={"db": "hdfc_core_banking", "region": "Mumbai"}
)
def fetch_statement(customer_id: str, month: str) -> dict:
    """Fetch statement from HDFC's core banking system."""
    # Simulated DB call
    return {
        "customer_id": customer_id,
        "month": month,
        "total_due_inr": 42850.00,
        "min_due_inr": 2142.00,
        "due_date": "2026-10-05",
        "transactions": [
            {"merchant": "Swiggy", "amount_inr": 8500, "date": "2026-09-12"},
            {"merchant": "Amazon", "amount_inr": 15000, "date": "2026-09-18"},
        ]
    }


@traceable(
    run_type="tool",
    name="check_fraud_signals",
    tags=["fraud", "security"]
)
def check_fraud_signals(transaction: dict) -> dict:
    """Check a transaction against fraud rules."""
    amount = transaction["amount_inr"]
    signals = []
    
    if amount > 5000:
        signals.append("high_value_transaction")
    if transaction["merchant"] == "Swiggy" and amount > 3000:
        signals.append("unusual_food_delivery_amount")
    
    return {
        "fraud_score": 0.82 if signals else 0.1,
        "signals": signals,
        "recommended_action": "flag_for_review" if signals else "approve"
    }


@traceable(run_type="chain", name="billing_dispute_pipeline")
def run_billing_dispute(customer_id: str, disputed_merchant: str):
    """Full dispute handling pipeline — custom, not LangGraph."""
    statement = fetch_statement(customer_id, "September-2026")
    
    # Find the disputed transaction
    disputed_txn = next(
        (t for t in statement["transactions"] if t["merchant"] == disputed_merchant),
        None
    )
    
    if not disputed_txn:
        return {"status": "transaction_not_found"}
    
    fraud_result = check_fraud_signals(disputed_txn)
    
    return {
        "disputed_amount_inr": disputed_txn["amount_inr"],
        "fraud_score": fraud_result["fraud_score"],
        "action": fraud_result["recommended_action"],
        "signals": fraud_result["signals"]
    }


# Run it — appears in LangSmith as a full trace tree
result = run_billing_dispute("HDFC-CUST-98234", "Swiggy")
print(result)

LangSmith trace:

``` markdown
Trace: billing_dispute_pipeline     [chain]
├── fetch_hdfc_statement            [tool]   55ms  → {total_due: 42850, ...}
└── check_fraud_signals             [tool]   3ms   → {fraud_score: 0.82, action: flag}
```

7. LLM Tracing — Inspecting the Prompt/Completion

The most valuable thing LangSmith does. Every llm run shows you:

``` markdown
Run: ChatOpenAI
├── INPUT
│   ├── system: "You are an HDFC billing specialist..."
│   └── human:  "My statement shows ₹8,500 from Swiggy I didn't make."
│
├── OUTPUT
│   └── "We have flagged this ₹8,500 transaction as potentially fraudulent.
│         A dispute has been raised (Ref: DIS-2026-88234). The amount will be
│         put on hold while we investigate (3-5 business days)."
│
└── METRICS
    ├── model:              gpt-4o
    ├── prompt_tokens:      112
    ├── completion_tokens:  54
    ├── total_tokens:       166
    ├── latency:            720ms
    └── estimated_cost:     $0.0019
```

**Note**: This is how we debug bad outputs — we see the exact prompt the model received.

8. Tool Tracing

When your LangGraph agent calls tools, LangSmith captures them as tool runs:

In [ ]:
from langchain_core.tools import tool
from langsmith import traceable

@tool
def get_reward_points(customer_id: str) -> dict:
    """Get HDFC SmartBuy reward point balance for a customer."""
    # Simulated API call to HDFC rewards system
    return {
        "customer_id": customer_id,
        "points_balance": 12450,
        "value_inr": 3112.50,          # 1 point = ₹0.25
        "expiring_this_month": 500,
        "tier": "Platinum"
    }


@tool
def raise_dispute(customer_id: str, amount_inr: float, merchant: str) -> dict:
    """Raise a billing dispute for a transaction."""
    dispute_id = f"DIS-2026-{hash(customer_id + merchant) % 100000:05d}"
    return {
        "dispute_id": dispute_id,
        "status": "raised",
        "expected_resolution_days": 5,
        "refund_method": "credit_to_card"
    }

LangSmith tool run view:

``` markdown
Run: get_reward_points              [tool]   45ms
├── INPUT:  {"customer_id": "HDFC-CUST-98234"}
└── OUTPUT: {"points_balance": 12450, "value_inr": 3112.50, "tier": "Platinum"}

Run: raise_dispute                  [tool]   120ms
├── INPUT:  {"customer_id": "HDFC-CUST-98234", "amount_inr": 8500.0, "merchant": "Swiggy"}
└── OUTPUT: {"dispute_id": "DIS-2026-88234", "status": "raised", "resolution_days": 5}
```

9. Retriever Tracing

For RAG pipelines, LangSmith traces the retriever and shows you which documents were returned.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Build a simple knowledge base (HDFC card policy docs)
docs = [
    Document(page_content="HDFC Regalia card offers 4 reward points per ₹150 spent.", 
             metadata={"source": "regalia_policy.pdf", "page": 3}),
    Document(page_content="Disputes must be raised within 60 days of the transaction date.", 
             metadata={"source": "dispute_policy.pdf", "page": 1}),
    Document(page_content="International transactions are charged 3.5% forex markup fee.", 
             metadata={"source": "fees_schedule.pdf", "page": 7}),
    Document(page_content="HDFC Millennia card offers 5% cashback on Amazon, Flipkart, and Swiggy.", 
             metadata={"source": "millennia_policy.pdf", "page": 2}),
]

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# This retriever call is automatically traced in LangSmith
results = retriever.invoke("What cashback do I get on Swiggy orders?")
for doc in results:
    print(doc.page_content)

LangSmith retriever run:

``` markdown
Run: VectorStoreRetriever           [retriever]   85ms
├── INPUT:  "What cashback do I get on Swiggy orders?"
└── OUTPUT: [
│     Document 1: "HDFC Millennia card offers 5% cashback on Amazon, Flipkart, and Swiggy."
│               metadata: {source: millennia_policy.pdf, page: 2}
│     Document 2: "HDFC Regalia card offers 4 reward points per ₹150 spent."
│               metadata: {source: regalia_policy.pdf, page: 3}
│   ]
```


**Why this matters**: If your RAG bot gives wrong answers, you open the retriever run and check — did it retrieve the right docs? If not, your embedding or chunking is the problem, not the LLM.

10. Trace Filtering — Finding Problems at Scale

In LangSmith's UI you can filter runs with a powerful query language.

10.1 Common Filter Queries

``` text
# Find all failed runs
error IS NOT NULL

# Find slow runs (>2 seconds)
latency > 2000

# Find high-cost runs
total_tokens > 2000

# Find by metadata
metadata.customer_tier = "Regalia"

# Find by tag
"fraud_query" in tags

# Find runs from a specific node
name = "billing"

# Combine filters
name = "billing" AND latency > 1500 AND error IS NOT NULL
```

10.2 Programmatic Filtering — LangSmith SDK

``` python
from langsmith import Client

client = Client()

# List recent failed runs in your project
failed_runs = client.list_runs(
    project_name="reliance_support_agent_dev",
    filter='error IS NOT NULL',
    run_type="chain",
    limit=50
)

for run in failed_runs:
    print(f"Run: {run.name}")
    print(f"Error: {run.error}")
    print(f"Input: {run.inputs}")
    print(f"Duration: {run.end_time - run.start_time}")
    print("---")
```

``` python
# Find all runs where the billing node was slow
slow_billing_runs = client.list_runs(
    project_name="hdfc_card_bot_prod",
    filter='name = "billing" AND latency > 2000',
    limit=20
)
```

``` python
# Find runs by metadata value
regalia_runs = client.list_runs(
    project_name="hdfc_card_bot_prod",
    filter='metadata.customer_tier = "Regalia"',
    limit=100
)
```

11. Production Debugging — The Full Workflow

Here's the complete debugging loop the curriculum describes:

``` text
LangGraph Agent
      ↓
LangSmith
      ↓
Trace
      ↓
Find bottleneck    → Sort by latency → billing node is 4x slower
      ↓
Find failed node   → Filter error IS NOT NULL → "tool get_invoice: KeyError 'amount'"
      ↓
Inspect input/output → billing node got customer_id=None (upstream bug in classify)
      ↓
Fix               → classify node now validates customer_id before routing
```


11.1 Example: Catching a Tool Failure

In [ ]:
from langsmith import Client
from langsmith.schemas import Run

client = Client()

def debug_failed_traces(project_name: str, limit: int = 10):
    """Find and diagnose failed traces."""
    
    # Step 1: Get root-level failed traces
    failed_traces = list(client.list_runs(
        project_name=project_name,
        filter='error IS NOT NULL',
        is_root=True,
        limit=limit
    ))
    
    print(f"Found {len(failed_traces)} failed traces\n")
    
    for trace in failed_traces:
        print(f"=== Trace: {trace.id} ===")
        print(f"Start: {trace.start_time}")
        print(f"Error: {trace.error}")
        print(f"Input query: {trace.inputs.get('query', 'N/A')}")
        
        # Step 2: Get all child runs of this trace
        child_runs = list(client.list_runs(
            project_name=project_name,
            trace_id=trace.id
        ))
        
        # Step 3: Find the specific node that failed
        failed_children = [r for r in child_runs if r.error]
        for child in failed_children:
            print(f"\n  Failed node: {child.name}  (type: {child.run_type})")
            print(f"  Input:  {child.inputs}")
            print(f"  Error:  {child.error}")
        
        print()


debug_failed_traces("hdfc_card_bot_prod")

Output:

``` markdown
Found 3 failed traces

=== Trace: a8f3b2c1-... ===
Start: 2026-09-19 09:14:22
Error: KeyError: 'amount_inr'
Input query: "What is my minimum due this month?"

  Failed node: get_invoice  (type: tool)
  Input:  {"customer_id": None, "month": "September-2026"}
  Error:  KeyError: 'amount_inr'
```

Now you know exactly what failed: get_invoice was called with customer_id=None. The bug is upstream in the node that sets customer_id.

11.2 Example: Finding Latency Bottlenecks

In [ ]:
def find_latency_bottlenecks(project_name: str):
    """Find which nodes are slowest."""
    from collections import defaultdict
    import statistics
    
    runs = list(client.list_runs(
        project_name=project_name,
        run_type="chain",
        limit=200
    ))
    
    # Group latencies by node name
    latencies = defaultdict(list)
    for run in runs:
        if run.end_time and run.start_time:
            duration_ms = (run.end_time - run.start_time).total_seconds() * 1000
            latencies[run.name].append(duration_ms)
    
    # Report
    print(f"{'Node':<25} {'Calls':>6} {'Avg ms':>8} {'P95 ms':>8} {'Max ms':>8}")
    print("-" * 60)
    
    for name, durations in sorted(latencies.items(), key=lambda x: -statistics.mean(x[1])):
        if len(durations) < 2:
            continue
        print(
            f"{name:<25} {len(durations):>6} "
            f"{statistics.mean(durations):>8.0f} "
            f"{sorted(durations)[int(len(durations)*0.95)]:>8.0f} "
            f"{max(durations):>8.0f}"
        )


find_latency_bottlenecks("hdfc_card_bot_prod")

Output:

``` markdown
Node                      Calls    Avg ms   P95 ms   Max ms
------------------------------------------------------------
handle_block_card            45      1820     3100     5200
handle_billing              112       890     1600     2800
classify                    200       480      820     1200
handle_rewards               43       420      700      980
```

**Immediately visible**: handle_block_card is 4× slower than handle_rewards. You open those traces and find the system prompt is too long — causing more tokens and slower completions.

12. Adding Feedback to Runs

You can programmatically log feedback on runs — useful for tracking thumbs up/down from users or automated eval scores.

In [ ]:
from langsmith import Client

client = Client()

def log_user_feedback(run_id: str, score: float, comment: str = ""):
    """Log user feedback on a trace."""
    client.create_feedback(
        run_id=run_id,
        key="user_satisfaction",       # metric name
        score=score,                    # 0.0 to 1.0
        comment=comment
    )

def log_correctness_score(run_id: str, is_correct: bool, evaluator: str = "human"):
    """Log correctness evaluation."""
    client.create_feedback(
        run_id=run_id,
        key="correctness",
        score=1.0 if is_correct else 0.0,
        source_info={"evaluator": evaluator}
    )


# After a user rates the response
log_user_feedback(
    run_id="a8f3b2c1-...",
    score=0.2,
    comment="Bot gave wrong refund timeline — said 3 days but should be 5-7 business days"
)

# In LangSmith UI, you can then filter runs by feedback score and 
# see which query types get the lowest ratings.

##### Mental Model — The Debugging Loop

``` text
1. Something is wrong in production
         ↓
2. Open LangSmith → your prod project
         ↓
3. Filter: error IS NOT NULL  →  find failed traces
         ↓
4. Click a failed trace → see the waterfall
         ↓
5. Find the red span (failed run)
         ↓
6. Click it → see exact input + error message
         ↓
7. Look at parent span → what state was passed in?
         ↓
8. Trace back to root → find where bad state originated
         ↓
9. Check LLM run above it → did the model output bad JSON? Wrong routing?
         ↓
10. Fix the node / prompt / tool and redeploy
         ↓
11. Filter by latency to find next bottleneck
         ↓
12. Repeat
```